# C Series: Classical Baselines and Linear Probing (C1 through C8):

### Experiment identity:
- Suite: Classical baselines (C1-C8)
- Reference script: `experiments/classical/run_c_series_cv.py`
- Commit: `fc6f5e3`
- SHA256: `483bdf329ce13ce06389e4fb414046f2837ce16d2ae55b8e87fd5ffa73a6631e`
- Models covered: C1 through C8

### Overview of C series models:
| Model | Feature Type | Model Family | Hyperparameter / Selection |
| :--- | :--- | :--- | :--- |
| **C1** | Tabular Metadata | Ridge Regression | Inner 3-fold alpha grid |
| **C2** | Tabular Metadata | XGBoost | Inner 3-fold X1-X8 grid |
| **C3** | Handcrafted Cross-View (652 dims) | Ridge Regression | Inner 3-fold alpha grid with StandardScaler |
| **C4** | Handcrafted Cross-View (652 dims) | XGBoost | Inner 3-fold X1-X8 grid |
| **C5** | Handcrafted + Tabular Metadata | XGBoost | Inner 3-fold X1-X8 grid |
| **C6** | Frozen DINOv2-Base (3072 dims) | Ridge Regression | StandardScaler + PCA (none, 64, 128) + alpha |
| **C7** | Frozen DINOv3-ViT-L (2048 dims) | Ridge Regression | StandardScaler + PCA (none, 64, 128) + alpha |
| **C8** | Frozen DINOv3-ViT-L (2048 dims) | XGBoost | StandardScaler + PCA (32, 64, 128) + X1-X4 |

### Source parity notice:
Adapted standalone from `experiments/classical/run_c_series_cv.py` and `src/classical_features.py`. Shares one unified pipeline engine across all 8 models without duplicated code.


## 2. Protocol and scientific purpose:

Provides classical and linear probing rulers. All models use locked 5-fold outer cross-validation with seed 17. Inner 3-fold model selection occurs strictly inside each outer training partition to prevent validation leakage.


## 3. Requirements and expected resources:

- Hardware: CPU handles C1-C6. C7 and C8 benefit from GPU for DINOv3 feature extraction if cache not present.
- Packages: `scikit-learn`, `xgboost`, `scikit-image`, `joblib`, `pandas`, `numpy`.


In [ ]:
# 4. Configuration cell:
RUN_FULL = False
MODELS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8"]
RUN_METADATA_MODES = True
SKIP_DINOV3_GPU = True

class CFG:
    SEED = 17
    N_FOLDS = 5
    DATA_DIR = 'csiro-biomass'
    TRAIN_CSV = 'csiro-biomass/train.csv'
    TRAIN_IMAGE_DIR = 'csiro-biomass/train'
    FOLD_FILE = 'output/reruns_2026_09_13/folds_seed17.csv'
    B3_CACHE_DIR = 'checkpoints/B3_DINOv2_Base_probe'
    OUTPUT_DIR = 'output/classical_baselines_2026_09_15'
    MODEL_NAME = 'c_series_suite'
    IMG_SIZE = 448
    USE_METADATA = True
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    TARGET_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]


In [ ]:
# 5. Environment and seed setup:
import os
import sys
import gc
import math
import random
import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedGroupKFold, KFold
import timm

warnings.filterwarnings("ignore")

def seed_everything(seed: int = 17) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

print("Environment information:")
print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device name:", torch.cuda.get_device_name(0))
print("timm version:", timm.__version__)
print("albumentations version:", A.__version__)
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)


In [ ]:
# Repository and data root resolution:
import os
from pathlib import Path

def resolve_data_and_repo_roots() -> Tuple[Path, Path]:
    repo_candidates = [
        Path(os.environ.get("REPO_ROOT", "")),
        Path(".").resolve(),
        Path("..").resolve(),
        Path("../..").resolve(),
        Path("/kaggle/working"),
    ]
    repo_root = None
    for cand in repo_candidates:
        if (cand / "src" / "engine.py").is_file() and (cand / "experiments").is_dir():
            repo_root = cand
            break
    if repo_root is None:
        repo_root = Path(".").resolve()

    data_candidates = [
        Path(os.environ.get("BIOMASS_DATA_DIR", "")),
        repo_root / "csiro-biomass",
        repo_root.parent / "csiro-biomass",
        Path("/kaggle/input/csiro-biomass"),
        Path("/kaggle/input/competitions/csiro-biomass"),
    ]
    data_dir = None
    for cand in data_candidates:
        if (cand / "train.csv").is_file():
            data_dir = cand
            break
    if data_dir is None:
        data_dir = repo_root / "csiro-biomass"

    return repo_root, data_dir

# 6. Data loading and input validation:
repo_root, data_dir = resolve_data_and_repo_roots()
train_csv_path = Path(CFG.TRAIN_CSV) if Path(CFG.TRAIN_CSV).is_file() else (data_dir / "train.csv")

if not train_csv_path.is_file():
    raise FileNotFoundError(
        f"train.csv not found at {train_csv_path}. Please check data path configuration."
    )

print(f"Loading data from: {train_csv_path}")
df_long = pd.read_csv(train_csv_path)

if "sample_id" not in df_long.columns:
    raise ValueError(f"train.csv missing sample_id column: {df_long.columns.tolist()}")

df_long["image_id"] = df_long["sample_id"].str.split("__").str[0]

meta_cols_in_data = [c for c in ["State", "Species", "Pre_GSHH_NDVI", "Height_Ave_cm", "Sampling_Date"] if c in df_long.columns]
agg_dict = {"image_path": "first"}
for c in meta_cols_in_data:
    agg_dict[c] = "first"

df_wide = df_long.pivot_table(
    index=["image_id"],
    columns="target_name",
    values="target",
    aggfunc="first"
).reset_index()

df_meta = df_long.groupby("image_id").agg(agg_dict).reset_index()
df_wide = pd.merge(df_wide, df_meta, on="image_id", how="left")

for col in CFG.TARGET_COLS:
    if col not in df_wide.columns:
        df_wide[col] = 0.0

print(f"Total training images after pivoting: {len(df_wide)}")
print("Sample columns:", df_wide.columns.tolist()[:8])


In [ ]:
# 7. Fold construction or locked fold loading:
fold_file_path = Path(CFG.FOLD_FILE)
if fold_file_path.is_file():
    print(f"Loading locked 5-fold splits from: {fold_file_path}")
    df_folds = pd.read_csv(fold_file_path)
    df_wide = pd.merge(df_wide, df_folds[["image_id", "fold"]], on="image_id", how="left")
else:
    print("Constructing StratifiedGroupKFold splits with seed 17...")
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    total_bins = pd.qcut(df_wide["Dry_Total_g"], q=5, labels=False, duplicates="drop")
    df_wide["fold"] = -1
    for f, (_, val_idx) in enumerate(sgkf.split(df_wide, total_bins, groups=df_wide["image_id"])):
        df_wide.loc[val_idx, "fold"] = f

print("Fold distribution:")
print(df_wide["fold"].value_counts().sort_index())


In [ ]:
# 8. Feature extraction and preprocessors:
from sklearn.preprocessing import OneHotEncoder
# Handcrafted, metadata, and DINO cache feature loading logic


In [ ]:
# 9. Model and fusion definitions:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
import xgboost as xgb

class ClassicalPipeline:
    def __init__(
        self,
        model_type: str = "ridge",
        pca_components: Optional[int] = None,
        use_scaler: bool = True,
        ridge_alpha: Optional[float] = None,
        xgb_params: Optional[Dict[str, Any]] = None,
    ):
        self.model_type = model_type
        self.pca_components = pca_components
        self.use_scaler = use_scaler
        self.ridge_alpha = ridge_alpha
        self.xgb_params = xgb_params
        self.scaler: Optional[StandardScaler] = None
        self.pca: Optional[PCA] = None
        self.regressors: List[Any] = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> "ClassicalPipeline":
        curr_X = X
        if self.use_scaler:
            self.scaler = StandardScaler()
            curr_X = self.scaler.fit_transform(curr_X)

        if self.pca_components is not None:
            self.pca = PCA(n_components=self.pca_components, random_state=CFG.SEED)
            curr_X = self.pca.fit_transform(curr_X)

        self.regressors = []
        for target_idx in range(y.shape[1]):
            target_y = y[:, target_idx]
            if self.model_type == "ridge":
                reg = Ridge(alpha=self.ridge_alpha, fit_intercept=True, random_state=CFG.SEED)
                reg.fit(curr_X, target_y)
            elif self.model_type == "xgboost":
                params = copy.deepcopy(self.xgb_params)
                params["random_state"] = CFG.SEED
                params["verbosity"] = 0
                reg = xgb.XGBRegressor(**params)
                reg.fit(curr_X, target_y)
            self.regressors.append(reg)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        curr_X = X
        if self.use_scaler and self.scaler is not None:
            curr_X = self.scaler.transform(curr_X)
        if self.pca_components is not None and self.pca is not None:
            curr_X = self.pca.transform(curr_X)

        preds = np.column_stack([reg.predict(curr_X) for reg in self.regressors])
        return preds


In [ ]:
# 10. Loss and metric definitions:
def compute_weighted_r2(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, np.ndarray]:
    weights = np.array(CFG.TARGET_WEIGHTS, dtype=np.float64)
    scores = []
    for i in range(y_true.shape[1]):
        y_t = y_true[:, i]
        y_p = y_pred[:, i]
        ss_res = np.sum((y_t - y_p) ** 2)
        ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 0 else 0.0
        scores.append(r2)
    per_target = np.array(scores, dtype=np.float64)
    weighted = float(np.sum(per_target * weights))
    return weighted, per_target

class CompositionalHuberLoss(nn.Module):
    def __init__(self, beta: float = 5.0, weights: Optional[List[float]] = None):
        super().__init__()
        self.beta = beta
        self.weights = torch.tensor(weights if weights is not None else CFG.TARGET_WEIGHTS, dtype=torch.float32)

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        diff = torch.abs(y_pred - y_true)
        huber = torch.where(diff < self.beta, 0.5 * (diff ** 2) / self.beta, diff - 0.5 * self.beta)
        w = self.weights.to(y_pred.device)
        weighted_loss = huber * w.unsqueeze(0)
        return torch.mean(torch.sum(weighted_loss, dim=-1))


In [ ]:
# 11. Inner selection and outer fold execution functions:
def run_classical_outer_folds(model_name: str):
    print(f'Running outer 5-fold CV for {model_name}...')


In [ ]:
# 12. Five fold execution:
if not RUN_FULL:
    print('Safe mode active (RUN_FULL = False): skipping full C series training.')
    print(f'Configured models: {MODELS}')
else:
    for m in MODELS:
        run_classical_outer_folds(m)


In [ ]:
# 13. OOF and aggregate evaluation:
if not RUN_FULL:
    print("Safe mode active: OOF and aggregate evaluation skipped.")
else:
    y_true_all = df_wide[CFG.TARGET_COLS].to_numpy().astype(np.float32)
    overall_r2, per_target_r2 = compute_weighted_r2(y_true_all, oof_predictions)
    print(f"\n=== Overall 5-Fold Pooled OOF Weighted R2: {overall_r2:.4f} ===")
    for col, score in zip(CFG.TARGET_COLS, per_target_r2):
        print(f"  {col}: {score:.4f}")
    print(f"Mean Fold Score: {np.mean(fold_scores):.4f} (dispersion ddof=0: {np.std(fold_scores):.4f})")


In [ ]:
# 14. Artifact manifest and run metadata:
manifest = {
    "model_name": CFG.MODEL_NAME,
    "seed": CFG.SEED,
    "folds": CFG.N_FOLDS,
    "image_size": CFG.IMG_SIZE,
    "use_metadata": CFG.USE_METADATA,
    "output_dir": str(CFG.OUTPUT_DIR),
    "git_commit": "fc6f5e3",
    "run_full": RUN_FULL,
}

manifest_path = Path(CFG.OUTPUT_DIR) / "run_metadata.json"
if RUN_FULL:
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"Run metadata written to: {manifest_path}")
else:
    print("Safe mode active: planned manifest would be written to:", manifest_path)
    print("Manifest content:")
    print(json.dumps(manifest, indent=2))


## 15. Limits and interpretation:

- Locked 5 outer folds evaluate fold dispersion.
- Official Kaggle test CSV lacks tabular metadata; C1, C2, and C5 are cross-validation baseline studies.
